# 🏭 W5: KPI 불량률 분석 & AI 도입 전/후 비교
**hexa-1: 제조업 AX 마스터클래스 — Week 5**

> ⏱️ 예상 소요시간: 60분 | 🎯 결과물: KPI 개선 보고서 + 차트

## Step 0: 환경 설정 (최초 1번만)

In [ ]:
# ─── 한국어 폰트 설치 (Colab 최초 1회) ───
import subprocess, sys
subprocess.run(['apt-get', '-qq', '-y', 'install', 'fonts-nanum'], capture_output=True)

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
_n=[f for f in fm.findSystemFonts() if 'Nanum' in f]
if _n: fm.fontManager.addfont(_n[0]); plt.rcParams['font.family']=fm.FontProperties(fname=_n[0]).get_name()
    plt.rcParams['font.family'] = fm.FontProperties(fname=nanum).get_name()
plt.rcParams['axes.unicode_minus'] = False
print('✅ 폰트 설정 완료:', plt.rcParams['font.family'])
!pip install -q pandas
import pandas as pd
print('✅ 준비 완료')

## Step 1: 공장 정보 입력 (✏️ 여기만 수정)

In [ ]:
FACTORY_INFO = {
    'name': '✏️ [교육 기업명]',
    'product': '✏️ [주요 제품명]',
    'ai_intro_date': '2025-07-01',  # AI 도입 시작일
    'target_defect_rate': 2.0,       # 목표 불량률 (%)
}
print('✅ 공장 정보:', FACTORY_INFO)


## Step 2: KPI 함수 로드

In [ ]:
# kpi_calculator 인라인 함수 (Colab에서 외부 scripts/ 폴더 불필요)
def calc_defect_rate(total, defective):
    if total <= 0: return 0.0
    return round((defective / total) * 100.0, 2)

print('✅ KPI 함수 로드 완료')


## Step 3: 데이터 로드 (GitHub 샘플 또는 직접 업로드)

In [ ]:
import io, requests

url = 'https://raw.githubusercontent.com/Reasonofmoon/hexa-1/main/shared/manufacturing_kpi_sample.csv'
try:
    df = pd.read_csv(url, encoding='utf-8-sig')
    print(f'✅ 샘플 데이터 로드: {len(df)}행')
except Exception as e:
    print(f'GitHub 로드 실패: {e} → 파일 업로드 모드')
    from google.colab import files
    uploaded = files.upload()
    df = pd.read_csv(io.BytesIO(list(uploaded.values())[0]), encoding='utf-8-sig')

# 컬럼 자동 감지
date_col   = next((c for c in df.columns if '날짜' in c or c.lower()=='date'), df.columns[0])
total_col  = next((c for c in df.columns if '생산량' in c or c.lower()=='total'), None)
defect_col = next((c for c in df.columns if '불량_수' in c or c.lower() in ('defective','defect')), None)
print(f'컬럼 → 날짜:{date_col}, 생산:{total_col}, 불량:{defect_col}')
df[date_col] = pd.to_datetime(df[date_col])
df.head(3)


## Step 4: 불량률 분석

In [ ]:
cutoff = pd.to_datetime(FACTORY_INFO['ai_intro_date'])
before = df[df[date_col] < cutoff]
after  = df[df[date_col] >= cutoff]

rate_before = calc_defect_rate(before[total_col].sum(), before[defect_col].sum())
rate_after  = calc_defect_rate(after[total_col].sum(),  after[defect_col].sum())
improvement = round(rate_before - rate_after, 2)
target = FACTORY_INFO['target_defect_rate']

print(f'📊 AI 도입 전 불량률: {rate_before}%')
print(f'📊 AI 도입 후 불량률: {rate_after}%')
print(f'📈 개선: -{improvement}%p')
print(f'목표({target}%): {"✅ 달성" if rate_after <= target else "🔄 진행중"}')


## Step 5: 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(['도입 전', '도입 후'], [rate_before, rate_after],
            color=['#e74c3c', '#2ecc71'], width=0.5)
axes[0].axhline(target, color='orange', linestyle='--', label=f'목표({target}%)')
axes[0].set_title('불량률 비교 (%)')
axes[0].set_ylabel('불량률 (%)')
axes[0].legend()

df['불량률'] = df[defect_col] / df[total_col] * 100
df.set_index(date_col)['불량률'].plot(ax=axes[1], title='불량률 추이', color='steelblue')
axes[1].axvline(cutoff, color='red', linestyle='--', label='AI 도입')
axes[1].legend()

plt.tight_layout()
plt.savefig('kpi_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 차트 저장 완료')


## Step 6: 보고서 다운로드

In [ ]:
import datetime
report = f"""# {FACTORY_INFO['name']} AI 도입 성과 보고서
제품: {FACTORY_INFO['product']}
보고일: {datetime.date.today()}

## 핵심 성과
| 구분 | 도입 전 | 도입 후 | 개선 |
|---|---|---|---|
| 불량률 | {rate_before}% | {rate_after}% | -{improvement}%p |
| 목표 달성 | - | {'✅' if rate_after <= target else '🔄'} | - |
"""
with open('kpi_report.md', 'w', encoding='utf-8') as f:
    f.write(report)

from google.colab import files
files.download('kpi_report.md')
files.download('kpi_chart.png')
print('✅ 다운로드 완료!')


---
## 🔥 확장 과제
1. `ai_intro_date`를 바꿔보고 결과 변화 확인
2. 내 공장 실제 CSV 업로드해서 실습
3. 목표 불량률을 낮춰가며 달성 시나리오 시뮬레이션

In [ ]:
# === Gemini AI 분석 (자동 추가됨) ===
!pip install -q google-generativeai
try:
    from google.colab import userdata; _api=userdata.get('GEMINI_API_KEY')
except: _api=input('GEMINI_API_KEY: ')
import google.generativeai as genai; genai.configure(api_key=_api)
# === 2026년 3월 기준 최신 모델 선택 ===
# ✅ GA(안정): gemini-2.5-flash  ← 기본값 (권장)
# 🔵 Preview : gemini-3.1-flash-lite-preview  (최신, 2026.3 출시)
# 🔵 Preview : gemini-3.1-pro-preview  (최강, 복잡한 분석용)
# ⚠️  구버전 gemini-2.0-flash 는 2026년 6월 1일 종료 예정
MODEL_NAME = 'gemini-2.5-flash'  # 원하는 모델로 변경하세요
_model=genai.GenerativeModel('gemini-2.5-flash'); print('✅ Gemini 분석 준비')


## 🤖 AI 인사이트 분석 (Gemini)

In [ ]:
_p=f"""제조업 AI 컨설턴트로서 제조 KPI 데이터를 분석하세요.
핵심 인사이트 3가지 + 즉시 개선 액션. 마크다운."""
_resp=_model.generate_content(_p)
print(_resp.text)
